# Notebook 2 — Feature Engineering & Traditional ML
**Member 2 Responsibility**

This notebook covers:
1. Hand-crafted features (semantic similarity, context–meaning alignment, etc.)
2. Training Ridge Regression and Gradient Boosting on these features
3. Evaluation on dev set — saves `ml_results.json`

**Prerequisites:** Run notebook1 first so `baselines_results.json` exists.
**Install:** `pip install sentence-transformers scikit-learn`

In [ ]:
import json
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle
import warnings
warnings.filterwarnings('ignore')

with open('train.json') as f:
    train_raw = json.load(f)
with open('dev.json') as f:
    dev_raw = json.load(f)

def to_df(raw):
    rows = []
    for sid, s in raw.items():
        rows.append({
            'id': sid,
            'homonym': s['homonym'],
            'judged_meaning': s['judged_meaning'],
            'precontext': s.get('precontext', ''),
            'sentence': s['sentence'],
            'ending': s.get('ending', '') or '',
            'average': s['average'],
            'stdev': s['stdev'],
            'example_sentence': s.get('example_sentence', '') or ''
        })
    return pd.DataFrame(rows)

train_df = to_df(train_raw)
dev_df   = to_df(dev_raw)
print(f'Train: {len(train_df)} | Dev: {len(dev_df)}')

## 1. Evaluation Helper

In [ ]:
def evaluate(preds, df):
    preds  = np.array(preds)
    trues  = df['average'].values
    stdevs = df['stdev'].values
    rho, _ = spearmanr(preds, trues)
    margin = np.maximum(stdevs, 1.0)
    acc    = (np.abs(preds - trues) <= margin).mean()
    return {'spearman_r': round(float(rho), 4), 'acc_within_stdev': round(float(acc), 4)}

## 2. Feature Engineering

We build a feature vector for each sample using:
- TF-IDF cosine similarity between story context and judged meaning
- TF-IDF cosine similarity between the homonym sentence and the meaning
- Word overlap features
- Length features (context length, meaning length)
- Whether an ending is present

In [ ]:
# ── Fit TF-IDF on all texts ───────────────────────────────────────────────
all_texts = (
    train_df['precontext'].tolist() +
    train_df['sentence'].tolist() +
    train_df['judged_meaning'].tolist() +
    train_df['ending'].tolist() +
    train_df['example_sentence'].tolist()
)
tfidf = TfidfVectorizer(max_features=15000, ngram_range=(1, 2), sublinear_tf=True)
tfidf.fit(all_texts)
print('TF-IDF fitted.')

def build_features(df):
    """
    Build a numeric feature matrix for a dataframe of story samples.
    Returns a numpy array of shape (n_samples, n_features).
    """
    features = []
    
    # TF-IDF vectors for each field
    full_context = (df['precontext'] + ' ' + df['sentence'] + ' ' + df['ending']).str.strip()
    precontext_vecs  = tfidf.transform(df['precontext'])
    sentence_vecs    = tfidf.transform(df['sentence'])
    ending_vecs      = tfidf.transform(df['ending'].replace('', 'none'))
    meaning_vecs     = tfidf.transform(df['judged_meaning'])
    example_vecs     = tfidf.transform(df['example_sentence'].replace('', 'none'))
    full_ctx_vecs    = tfidf.transform(full_context)

    for i in range(len(df)):
        row = df.iloc[i]
        
        # Cosine similarities (semantic alignment features)
        sim_fullctx_meaning  = cosine_similarity(full_ctx_vecs[i], meaning_vecs[i])[0][0]
        sim_sentence_meaning = cosine_similarity(sentence_vecs[i], meaning_vecs[i])[0][0]
        sim_precontext_meaning = cosine_similarity(precontext_vecs[i], meaning_vecs[i])[0][0]
        sim_ending_meaning   = cosine_similarity(ending_vecs[i], meaning_vecs[i])[0][0]
        sim_example_meaning  = cosine_similarity(example_vecs[i], meaning_vecs[i])[0][0]
        sim_sentence_example = cosine_similarity(sentence_vecs[i], example_vecs[i])[0][0]
        sim_fullctx_example  = cosine_similarity(full_ctx_vecs[i], example_vecs[i])[0][0]

        # Word overlap: homonym in meaning?
        homonym = row['homonym'].lower()
        meaning_lower = row['judged_meaning'].lower()
        ctx_lower = (row['precontext'] + ' ' + row['sentence']).lower()

        homonym_in_meaning  = float(homonym in meaning_lower)
        homonym_in_ctx      = float(homonym in ctx_lower)

        # Length features
        meaning_len  = len(row['judged_meaning'].split())
        context_len  = len(row['precontext'].split())
        sentence_len = len(row['sentence'].split())
        has_ending   = float(bool(row['ending']))
        ending_len   = len(row['ending'].split()) if row['ending'] else 0

        feat = [
            sim_fullctx_meaning,
            sim_sentence_meaning,
            sim_precontext_meaning,
            sim_ending_meaning,
            sim_example_meaning,
            sim_sentence_example,
            sim_fullctx_example,
            homonym_in_meaning,
            homonym_in_ctx,
            meaning_len,
            context_len,
            sentence_len,
            has_ending,
            ending_len,
        ]
        features.append(feat)

    return np.array(features)

print('Building train features...')
X_train = build_features(train_df)
y_train = train_df['average'].values

print('Building dev features...')
X_dev = build_features(dev_df)
y_dev = dev_df['average'].values

print(f'Feature matrix shape — Train: {X_train.shape}, Dev: {X_dev.shape}')

## 3. Ridge Regression

In [ ]:
ridge_model = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=1.0))
])
ridge_model.fit(X_train, y_train)

ridge_preds = ridge_model.predict(X_dev)
# Clip predictions to valid range [1, 5]
ridge_preds = np.clip(ridge_preds, 1, 5)

ridge_results = evaluate(ridge_preds, dev_df)
print('Ridge Regression:', ridge_results)

# Save model
with open('ridge_model.pkl', 'wb') as f:
    pickle.dump(ridge_model, f)
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
print('Models saved.')

## 4. Gradient Boosting Regressor

In [ ]:
gb_model = GradientBoostingRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42
)
gb_model.fit(X_train, y_train)

gb_preds = np.clip(gb_model.predict(X_dev), 1, 5)
gb_results = evaluate(gb_preds, dev_df)
print('Gradient Boosting:', gb_results)

with open('gb_model.pkl', 'wb') as f:
    pickle.dump(gb_model, f)
print('GB model saved.')

## 5. Feature Importance

In [ ]:
import matplotlib.pyplot as plt

feature_names = [
    'sim_fullctx_meaning', 'sim_sentence_meaning', 'sim_precontext_meaning',
    'sim_ending_meaning', 'sim_example_meaning', 'sim_sentence_example',
    'sim_fullctx_example', 'homonym_in_meaning', 'homonym_in_ctx',
    'meaning_len', 'context_len', 'sentence_len', 'has_ending', 'ending_len'
]

importances = gb_model.feature_importances_
sorted_idx  = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar([feature_names[i] for i in sorted_idx], importances[sorted_idx], color='steelblue')
ax.set_title('Gradient Boosting — Feature Importances')
ax.set_ylabel('Importance')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: feature_importance.png')

## 6. Save Results

In [ ]:
ml_results = {
    'Ridge Regression':     ridge_results,
    'Gradient Boosting':    gb_results,
}

import pandas as pd
print(pd.DataFrame(ml_results).T.to_string())

with open('ml_results.json', 'w') as f:
    json.dump(ml_results, f, indent=2)
print('Saved: ml_results.json')

## ✅ Notebook 2 Complete
Files produced:
- `ridge_model.pkl` — Ridge Regression model
- `gb_model.pkl` — Gradient Boosting model  
- `tfidf_vectorizer.pkl` — TF-IDF vectorizer
- `ml_results.json` — evaluation results
- `feature_importance.png`

Next: Run **notebook3_transformer_model.ipynb** (this is the main model)